# Neural Interventions: Optogenetics, Pharmacology, and Stimulation

Comprehensive guide to modeling neural interventions:
- **Optogenetics**: ChR2, NpHR, and other opsins with wavelength-specific activation
- **Pharmacology**: Drug dose-response curves and receptor dynamics
- **Neural stimulation**: TMS, DBS, tDCS with realistic physics

Use these tools to test causal hypotheses about foundation model mechanisms.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from neuros_mechint.interventions import (
    ChR2, NpHR, ChETA, ReaChR,
    OptoStimulator,
    Drugs,
    DBS, TMS, TDCS
)
from neuros_mechint.visualization import InterventionVisualizer

%matplotlib inline

## 1. Optogenetics

### Channelrhodopsin-2 (ChR2)

In [ ]:
# Create ChR2 opsin
chr2 = ChR2(g_max=1000.0, wavelength_peak=470.0)  # Blue light (470 nm)

# Simulate light-evoked response
dt = 0.1  # ms
duration = 200  # ms
n_steps = int(duration / dt)
time = np.arange(n_steps) * dt

# Light pulse (10 ms, 10 mW/mm²)
light_intensity = np.zeros(n_steps)
light_intensity[500:600] = 10.0  # 50-60 ms

# Membrane voltage (simplified)
V = -70.0 * torch.ones(1)
photocurrents = []
voltages = []

for i, intensity in enumerate(light_intensity):
    I_photo = chr2.photocurrent(V, intensity, dt)
    V += I_photo * 0.01  # Simple integration (real neuron would be more complex)
    
    photocurrents.append(float(I_photo))
    voltages.append(float(V))

# Visualize
viz = InterventionVisualizer()
fig = viz.plot_optogenetic_response(
    time=time,
    light_intensity=light_intensity,
    neural_response=np.array(voltages),
    opsin_name="ChR2",
    wavelength=470.0
)
fig.show()

print(f"Peak depolarization: {np.max(voltages) - voltages[0]:.2f} mV")
print(f"Time to peak: {time[np.argmax(voltages)]:.1f} ms")

### Comparing Multiple Opsins

In [ ]:
# Create different opsins
opsins = {
    'ChR2': ChR2(wavelength_peak=470.0),
    'ChETA': ChETA(wavelength_peak=470.0),  # Faster kinetics
    'ReaChR': ReaChR(wavelength_peak=590.0),  # Red-shifted
    'NpHR': NpHR(wavelength_peak=590.0)  # Inhibitory (chloride pump)
}

# Test all with same light pulse
light_pulse = np.zeros(1000)
light_pulse[200:300] = 10.0
time = np.arange(1000) * 0.1

responses = {}
for name, opsin in opsins.items():
    V = -70.0 * torch.ones(1)
    response = []
    
    for intensity in light_pulse:
        I = opsin.photocurrent(V, intensity, 0.1)
        response.append(float(I))
    
    responses[name] = response

# Plot comparison
plt.figure(figsize=(12, 6))

plt.subplot(2, 1, 1)
plt.fill_between(time, 0, light_pulse, alpha=0.3, label='Light')
plt.ylabel('Light (mW/mm²)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
for name, response in responses.items():
    plt.plot(time, response, label=name, linewidth=2)
plt.xlabel('Time (ms)')
plt.ylabel('Photocurrent (pA)')
plt.title('Opsin Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Kinetics comparison
for name, response in responses.items():
    r = np.array(response)
    onset_idx = np.where(r > r.max() * 0.1)[0][0]
    peak_idx = np.argmax(r)
    tau_on = (peak_idx - onset_idx) * 0.1
    print(f"{name}: τ_on = {tau_on:.1f} ms, peak = {r.max():.1f} pA")

## 2. Pharmacology

### Dose-Response Curves

In [ ]:
# Common neuroscience drugs
drugs = {
    'TTX': Drugs.TTX(),        # Sodium channel blocker
    'TEA': Drugs.TEA(),        # Potassium channel blocker
    'APV': Drugs.APV(),        # NMDA antagonist
    'CNQX': Drugs.CNQX(),      # AMPA antagonist
    'Gabazine': Drugs.Gabazine()  # GABA_A antagonist
}

# Generate dose-response curves
doses = np.logspace(-3, 2, 50)  # 0.001 to 100 μM

plt.figure(figsize=(14, 5))

for i, (name, drug) in enumerate(drugs.items()):
    responses = [drug.dose_response(d) for d in doses]
    
    plt.subplot(1, 2, 1)
    plt.plot(doses, responses, label=f'{name} (EC50={drug.EC50:.2f})', linewidth=2)

plt.subplot(1, 2, 1)
plt.xscale('log')
plt.xlabel('Dose (μM)')
plt.ylabel('Normalized Response')
plt.title('Drug Dose-Response Curves')
plt.legend()
plt.grid(True, alpha=0.3)

# Focus on TTX
ttx = drugs['TTX']
ttx_responses = [ttx.dose_response(d) for d in doses]

# Fit parameters
fit_params = {
    'EC50': ttx.EC50,
    'hill_coefficient': ttx.hill_coefficient,
    'Emax': 1.0
}

plt.subplot(1, 2, 2)
viz = InterventionVisualizer()
# Note: This would normally create a plotly figure, so we'll use matplotlib
plt.semilogx(doses, ttx_responses, 'o', markersize=8, label='Data')

# Hill equation fit
fitted = [fit_params['Emax'] * (d**fit_params['hill_coefficient']) / 
          (fit_params['EC50']**fit_params['hill_coefficient'] + d**fit_params['hill_coefficient'])
          for d in doses]
plt.semilogx(doses, fitted, 'r--', linewidth=2, label='Hill fit')
plt.axvline(fit_params['EC50'], color='g', linestyle=':', label=f"EC50={fit_params['EC50']:.2f}")
plt.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('TTX Concentration (μM)')
plt.ylabel('Na+ Channel Block')
plt.title('TTX Dose-Response Detail')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Neural Stimulation

### Deep Brain Stimulation (DBS)

In [ ]:
# Create DBS stimulator (typical PD parameters)
dbs = DBS(
    frequency=130.0,  # Hz
    amplitude=3.0,    # V
    pulse_width=0.06  # ms
)

# Simulate 100 ms
dt = 0.01  # ms (fine resolution for pulse)
duration = 100  # ms
n_steps = int(duration / dt)
time = np.arange(n_steps) * dt

# Get stimulation current
currents = [dbs.stimulate(t) for t in time]

# Plot
plt.figure(figsize=(12, 8))

# Full trace
plt.subplot(3, 1, 1)
plt.plot(time, currents, 'b', linewidth=1)
plt.ylabel('Current (mA)')
plt.title(f'DBS Waveform ({dbs.frequency} Hz, {dbs.amplitude} V, {dbs.pulse_width} ms pulse)')
plt.grid(True, alpha=0.3)

# Zoomed view of single pulse
plt.subplot(3, 1, 2)
zoom_start = int(10 / dt)
zoom_end = int(20 / dt)
plt.plot(time[zoom_start:zoom_end], currents[zoom_start:zoom_end], 'b', linewidth=2)
plt.ylabel('Current (mA)')
plt.title('Single Pulse Detail')
plt.grid(True, alpha=0.3)

# Frequency spectrum
plt.subplot(3, 1, 3)
from scipy import signal
freqs, psd = signal.welch(currents, fs=1/(dt/1000), nperseg=1024)
plt.semilogy(freqs, psd, 'b', linewidth=2)
plt.axvline(dbs.frequency, color='r', linestyle='--', label=f'Stim freq: {dbs.frequency} Hz')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power Spectral Density')
plt.title('Stimulation Spectrum')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim([0, 500])

plt.tight_layout()
plt.show()

### Transcranial Magnetic Stimulation (TMS)

Model electric field induced by TMS coil.

In [ ]:
# Create TMS stimulator
tms = TMS(
    intensity=100.0,      # % motor threshold
    coil_position=(0, 0, 50),  # 50 mm above origin
    coil_orientation=(0, 0, -1)  # Pointing down
)

# Calculate field at grid of points (cortical surface)
x = np.linspace(-50, 50, 50)  # mm
y = np.linspace(-50, 50, 50)
X, Y = np.meshgrid(x, y)
Z = np.zeros_like(X)  # Cortical surface

field_strength = np.zeros_like(X)
for i in range(len(x)):
    for j in range(len(y)):
        point = np.array([X[i,j], Y[i,j], Z[i,j]])
        field_strength[i,j] = tms.calculate_field(point)

# Visualize
plt.figure(figsize=(10, 8))

plt.contourf(X, Y, field_strength, levels=20, cmap='hot')
plt.colorbar(label='Electric Field (V/m)')
plt.plot(0, 0, 'bx', markersize=15, markeredgewidth=3, label='Coil center')
plt.xlabel('X Position (mm)')
plt.ylabel('Y Position (mm)')
plt.title('TMS-Induced Electric Field')
plt.axis('equal')
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

print(f"Peak field strength: {field_strength.max():.1f} V/m")
print(f"Field at center: {field_strength[25, 25]:.1f} V/m")

## 4. Application to Foundation Models

Use interventions to test causal hypotheses.

In [ ]:
# Simulate foundation model with intervention
class SimpleFoundationModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = torch.nn.Linear(100, 100)
    
    def forward(self, x, intervention=None):
        h = self.layer(x)
        
        # Apply intervention
        if intervention is not None:
            h = intervention(h)
        
        return torch.relu(h)

model = SimpleFoundationModel()
input_data = torch.randn(10, 100)

# Define "optogenetic" intervention (amplify specific units)
def opto_intervention(activations):
    # Amplify units 20-30 (like ChR2 activation)
    amplified = activations.clone()
    amplified[:, 20:30] *= 2.0
    return amplified

# Define "pharmacological" intervention (block specific units)
def drug_intervention(activations):
    # Block units 40-50 (like TTX blocking Na+ channels)
    blocked = activations.clone()
    blocked[:, 40:50] *= 0.1
    return blocked

# Compare conditions
baseline = model(input_data, intervention=None)
opto_condition = model(input_data, intervention=opto_intervention)
drug_condition = model(input_data, intervention=drug_intervention)

# Visualize
plt.figure(figsize=(12, 4))

conditions = {
    'Baseline': baseline,
    'Optogenetic': opto_condition,
    'Pharmacological': drug_condition
}

for i, (name, activations) in enumerate(conditions.items()):
    plt.subplot(1, 3, i+1)
    plt.imshow(activations.detach().numpy().T, aspect='auto', cmap='viridis')
    plt.colorbar(label='Activation')
    plt.xlabel('Sample')
    plt.ylabel('Unit')
    plt.title(f'{name} Condition')
    
    # Highlight intervention zones
    if name == 'Optogenetic':
        plt.axhspan(20, 30, facecolor='cyan', alpha=0.3)
    elif name == 'Pharmacological':
        plt.axhspan(40, 50, facecolor='red', alpha=0.3)

plt.tight_layout()
plt.show()

# Quantify effects
opto_effect = (opto_condition.mean() - baseline.mean()) / baseline.mean()
drug_effect = (drug_condition.mean() - baseline.mean()) / baseline.mean()

print(f"\nIntervention Effects:")
print(f"Optogenetic: {opto_effect.item()*100:+.1f}% change")
print(f"Pharmacological: {drug_effect.item()*100:+.1f}% change")

## Conclusion

This notebook demonstrated:
1. **Optogenetics**: Multiple opsin variants with realistic kinetics
2. **Pharmacology**: Dose-response curves for common drugs
3. **Neural stimulation**: DBS and TMS with physical modeling
4. **Foundation model testing**: Applying interventions to test causality

These tools enable rigorous causal testing of foundation model mechanisms by simulating experimental interventions used in neuroscience.